# EigenGrooves — Latent Space Analysis

Explores the SVD decomposition, the latent audio dimensions it discovers, and
how the recommender behaves in that space.

This notebook runs on a **fresh clone with no dataset**: it falls back to the
synthetic catalogue. Point `DATA_PATH` at a real CSV to use one.

Requirements: `pip install -e ".[notebook]"`

In [ ]:
import sys
from pathlib import Path

# Work from a clone without installing the package.
_src = Path.cwd().parent / "src"
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import matplotlib.pyplot as plt
import numpy as np

from eigengrooves import (
    Catalog,
    Recommender,
    build_groups,
    build_standard_rankers,
    compare_rankers,
    fit_latent_model,
    format_comparison,
    make_synthetic_catalog,
)
from eigengrooves.linalg import svd
from eigengrooves.normalization import fit_scaler
from eigengrooves.rank import rank_by_elbow, rank_by_gavish_donoho, rank_by_variance

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
%matplotlib inline

## 1. Load the catalogue

`Catalog` deduplicates by `(track, artist)` on load. That matters: chart
datasets carry one row per track per week, and without collapsing them the
recommender returns the query track back to itself at similarity 1.0.

In [ ]:
DATA_PATH = Path.cwd().parent / "data" / "spotify_songs.csv"

if DATA_PATH.exists():
    catalog = Catalog.from_csv(DATA_PATH)
    print(f"Loaded {len(catalog)} unique tracks from {DATA_PATH.name}")
else:
    catalog = make_synthetic_catalog(n_songs=3000, random_state=20240)
    print(f"No dataset found — using {len(catalog)} synthetic tracks")

print(f"Duplicate/invalid rows collapsed: {catalog.n_duplicates_removed}")
print(f"Features: {', '.join(catalog.feature_names)}")
catalog.frame[list(catalog.feature_names)].describe().T

## 2. Why scaling is mandatory

`tempo` spans hundreds of BPM while `danceability` lives in [0, 1]. Without
scaling the decomposition is a study of tempo and nothing else.

In [ ]:
raw_variance = catalog.features.var(axis=0)
scaled, scaler = fit_scaler(catalog.features, method="zscore")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].barh(catalog.feature_names, raw_variance, color="indianred")
axes[0].set_xscale("log")
axes[0].set_title("Variance before scaling (log scale)")
axes[1].barh(catalog.feature_names, scaled.var(axis=0), color="steelblue")
axes[1].set_title("Variance after z-scoring")
plt.tight_layout()
plt.show()

dominant = catalog.feature_names[int(np.argmax(raw_variance))]
print(f"Unscaled, '{dominant}' alone holds {raw_variance.max() / raw_variance.sum():.1%} of total variance.")

## 3. The spectrum, and how many components to keep

Three principled answers rather than a hardcoded `k = 5`. Note the cumulative
curve: this is the number the original version reported as "100%" regardless
of `k`, because it normalised the truncated spectrum against itself.

In [ ]:
_, spectrum, _ = svd(scaled)
cumulative = np.cumsum(spectrum**2) / np.sum(spectrum**2)

k_variance = rank_by_variance(spectrum, 0.90)
k_elbow = rank_by_elbow(spectrum)
k_gd = rank_by_gavish_donoho(spectrum, *scaled.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(spectrum) + 1), spectrum, color="steelblue")
axes[0].set(xlabel="Component", ylabel="Singular value", title="Scree plot")

axes[1].plot(range(1, len(spectrum) + 1), cumulative * 100, "o-", color="coral")
axes[1].axhline(90, color="gray", ls="--", lw=1, label="90% threshold")
for k, colour, label in [
    (k_variance, "seagreen", f"variance → k={k_variance}"),
    (k_elbow, "purple", f"elbow → k={k_elbow}"),
    (k_gd, "crimson", f"Gavish–Donoho → k={k_gd}"),
]:
    axes[1].axvline(k, color=colour, ls=":", lw=2, label=label)
axes[1].set(xlabel="Components kept", ylabel="Cumulative variance (%)", title="Explained variance")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

for name, k in [("variance (90%)", k_variance), ("elbow", k_elbow), ("Gavish–Donoho", k_gd)]:
    print(f"{name:>16}: k={k}  retains {cumulative[k - 1]:.1%} of variance")

## 4. Backend accuracy: why one-sided Jacobi

The textbook route to an SVD is to eigendecompose $A^\mathsf{T}A$. That squares
the condition number, and audio features are strongly correlated, so the small
singular values arrive as noise. One-sided Jacobi never forms that product.

Reconstruction error will *not* show you this — it is dominated by the large
components. Relative error per singular value is the metric that discriminates.

In [ ]:
rng = np.random.default_rng(0)
base = rng.normal(size=(800, 4)) @ rng.normal(size=(4, 9))
ill = base + 1e-6 * rng.normal(size=(800, 9))
truth = np.linalg.svd(ill, compute_uv=False)

print(f"{'component':>10}  {'true':>12}  {'jacobi':>12}  {'eigh (A^T A)':>14}")
errors = {}
for backend in ("jacobi", "eigh"):
    _, sigma, _ = svd(ill, backend=backend)
    errors[backend] = np.abs(sigma - truth[: len(sigma)]) / truth[: len(sigma)]

_, s_jac, _ = svd(ill, backend="jacobi")
_, s_eig, _ = svd(ill, backend="eigh")
for i in range(len(truth)):
    print(f"{i + 1:>10}  {truth[i]:>12.6g}  {s_jac[i]:>12.6g}  {s_eig[i]:>14.6g}")

print(f"\nmax relative error — jacobi: {errors['jacobi'].max():.2e}")
print(f"max relative error — eigh  : {errors['eigh'].max():.2e}")

## 5. Fit the model and read the latent features

Signs are canonicalised (the largest loading in each component is forced
positive), so these interpretations are reproducible across runs and backends.

In [ ]:
model = fit_latent_model(catalog.features, catalog.feature_names, k="variance", random_state=0)
evr = model.explained_variance()

print(model.rank_selection)
print(f"Retained {evr.sum():.1%} of total variance across {model.k} of {len(model.full_spectrum)} components\n")
for i in range(1, model.k + 1):
    print(f"  LF{i}  (σ={model.singular_values[i - 1]:6.2f}, {evr[i - 1]:5.1%})  {model.describe_component(i, top_n=3)}")

In [ ]:
fig, axes = plt.subplots(1, model.k, figsize=(3.1 * model.k, 4), sharey=True)
axes = np.atleast_1d(axes)
for i, ax in enumerate(axes, start=1):
    weights = model.components[i - 1]
    ax.barh(
        catalog.feature_names,
        weights,
        color=["steelblue" if w >= 0 else "coral" for w in weights],
    )
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"LF{i}\nσ={model.singular_values[i - 1]:.1f} · {evr[i - 1]:.1%}", fontsize=9)
    ax.set_xlim(-1, 1)
plt.suptitle("Latent feature loadings (right singular vectors)", fontsize=12)
plt.tight_layout()
plt.show()

## 6. The catalogue in latent space

If the decomposition captures anything real, tracks should separate by genre
along the leading components without ever having been told what a genre is.

In [ ]:
latent = model.transform(catalog.features)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
if "genre" in catalog.frame.columns:
    genres = catalog.frame["genre"].astype(str)
    for genre in sorted(genres.unique()):
        mask = (genres == genre).to_numpy()
        axes[0].scatter(latent[mask, 0], latent[mask, 1], s=8, alpha=0.55, label=genre)
    axes[0].legend(fontsize=7, ncol=2, markerscale=2)
    axes[0].set_title("Latent space, coloured by genre (never shown to the model)")
else:
    axes[0].scatter(latent[:, 0], latent[:, 1], s=8, alpha=0.4, c="steelblue")
    axes[0].set_title("Latent space")
axes[0].set(xlabel="LF1", ylabel="LF2")

axes[1].scatter(scaled[:, 0], scaled[:, 1], s=8, alpha=0.35, c="gray")
axes[1].set(xlabel=catalog.feature_names[0], ylabel=catalog.feature_names[1],
            title="Two raw features, for comparison")
plt.tight_layout()
plt.show()

## 7. Recommendations, with explanations

Every result decomposes its similarity score into per-component contributions,
so the reasoning is inspectable rather than a bare number.

In [ ]:
recommender = Recommender(model, catalog)

# Seed with several tracks by one artist so the playlist has a direction.
by_artist = {}
for idx, artist in enumerate(catalog.artists):
    by_artist.setdefault(artist, []).append(idx)
seeds = max(by_artist.values(), key=len)[:4]

print("Playlist:")
for i in seeds:
    print(f"  · {catalog.describe(i)}")

for strategy in ("overall_top", "mmr"):
    print(f"\n=== {strategy} ===")
    for n, item in enumerate(recommender.recommend(seeds, n=6, strategy=strategy, explain=True), 1):
        print(f"  {n}. {item.title} — {item.artist}  [{item.score:.4f}]")
        print(f"     {item.explanation.summary()}")

## 8. Does the SVD actually earn its keep?

The premise of the project is that projecting to a latent subspace beats using
the raw features. That is a hypothesis, and `raw_cosine` is the control that
tests it. Run it and read the answer rather than assuming one.

In [ ]:
models = {
    f"svd_k{k}": fit_latent_model(catalog.features, catalog.feature_names, k=k, random_state=0)
    for k in (2, 3, 5, 7)
}
models["svd_k5_whiten"] = models["svd_k5"].with_whiten(True)

rankers = build_standard_rankers(catalog, scaled, models, random_state=0)
groups = build_groups(catalog, group_by="artist", seed_size=3, min_group_size=6, random_state=0)
results = compare_rankers(rankers, groups, catalog, scaled, k=10)

print(f"{len(groups)} held-out queries, grouped by artist\n")
print(format_comparison(results, k=10))

In [ ]:
names = [r.name for r in results]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, title in [
    (axes[0], "ndcg", "Accuracy — NDCG@10"),
    (axes[1], "diversity", "Intra-list diversity"),
]:
    values = [r.metrics[metric] for r in results]
    colours = ["crimson" if n == "raw_cosine" else "lightgray" if n in ("random", "popularity")
               else "steelblue" for n in names]
    ax.barh(names, values, color=colours)
    ax.invert_yaxis()
    ax.set_title(title)
plt.suptitle("Red = the no-SVD control the latent models must beat", fontsize=10)
plt.tight_layout()
plt.show()

best_svd = max((r for r in results if r.name.startswith("svd")), key=lambda r: r.metrics["ndcg"])
raw = next(r for r in results if r.name == "raw_cosine")
verdict = "beats" if best_svd.metrics["ndcg"] > raw.metrics["ndcg"] else "does NOT beat"
print(f"Best latent model ({best_svd.name}, NDCG {best_svd.metrics['ndcg']:.4f}) "
      f"{verdict} raw cosine (NDCG {raw.metrics['ndcg']:.4f}).")
print(f"Diversity: {best_svd.metrics['diversity']:.4f} vs {raw.metrics['diversity']:.4f}")

### Reading the result honestly

On the held-out-artist protocol, dimensionality reduction tends to **cost**
retrieval accuracy and **buy** diversity and catalogue coverage, with the
trade-off steepening as `k` falls. Every latent model still beats random and
popularity by a wide margin, so the features carry real signal — the question
is only whether compressing them helps.

Two caveats that matter for interpretation:

- The protocol rewards recovering *known stylistic neighbours*, which
  structurally penalises the cross-genre discovery this project is actually
  interested in. A good surprising recommendation scores as a miss.
- On the synthetic catalogue, the generative structure is known and influences
  the outcome. Re-run against a real dataset before treating any of it as
  settled.